# Semana 13: La Navaja Suiza - Descomposición en Valores Singulares (SVD)
## Notebook Conceptual (NB1) - Explorando SVD con Datos Sintéticos e Imágenes

### Propósito de la sesión
Descubrir la **descomposición en valores singulares (SVD)**, una de las herramientas más poderosas y estables del álgebra lineal. Exploraremos su interpretación geométrica, la aproximación de bajo rango, y aplicaciones prácticas como la **compresión de imágenes**, además de conectar con sistemas de recomendación, LSA, pseudoinversa y análisis de espacios latentes en IA.

### Objetivos de aprendizaje
*   Descomponer una matriz $\mathbf{A}$ en $\mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^\top$ usando `np.linalg.svd`.
*   Interpretar el significado de las matrices $\mathbf{U}$, $\boldsymbol{\Sigma}$ (valores singulares) y $\mathbf{V}^\top$.
*   Reconstruir la matriz usando solo los primeros $k$ valores singulares (aproximación de bajo rango).
*   Aplicar SVD a una imagen para comprimirla y observar la relación entre calidad y número de componentes.
*   Conectar con aplicaciones en IA: compresión de imágenes, filtrado colaborativo (RecSys), LSA (NLP), pseudoinversa (DL) y análisis de espacios latentes (GenAI).

## Configuración Inicial

Importamos las librerías necesarias: NumPy para operaciones, Matplotlib para visualizaciones y skimage para imágenes de ejemplo.

In [ ]:
# Importamos librerías
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, color

# Configuración de matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Fijamos semilla para reproducibilidad
np.random.seed(42)

print("🎯 Librerías importadas correctamente.")

---
## 1. La Descomposición en Valores Singulares (SVD)

El teorema de la SVD establece que cualquier matriz $\mathbf{A} \in \mathbb{R}^{m \times n}$ puede factorizarse como:

$$\mathbf{A} = \mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^\top$$

donde:
*   $\mathbf{U} \in \mathbb{R}^{m \times m}$ es una matriz ortogonal (sus columnas son los **vectores singulares izquierdos**).
*   $\boldsymbol{\Sigma} \in \mathbb{R}^{m \times n}$ es una matriz diagonal con los **valores singulares** $\sigma_1 \geq \sigma_2 \geq \dots \geq \sigma_r > 0$ en la diagonal (y ceros fuera).
*   $\mathbf{V} \in \mathbb{R}^{n \times n}$ es una matriz ortogonal (sus columnas son los **vectores singulares derechos**).

### 1.1. SVD de una matriz pequeña

In [ ]:
# Creamos una matriz pequeña (4x3)
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9],
              [10, 11, 12]], dtype=float)

print("Matriz A (4x3):\n", A)

# Calculamos la SVD (versión completa)
U, s, Vt = np.linalg.svd(A, full_matrices=True)

print("\n🔷 Matriz U (4x4):")
print(U)
print("\n🔶 Valores singulares (s):", s)
print("\n🔷 Matriz Vt (3x3):")
print(Vt)

# Reconstrucción
Sigma = np.zeros((4, 3))
Sigma[:3, :3] = np.diag(s)
A_reconstructed = U @ Sigma @ Vt

print("\n📌 Error de reconstrucción:", np.linalg.norm(A - A_reconstructed))

### 1.2. Interpretación de U, Σ y Vᵀ

*   Las **columnas de U** son una base ortonormal para el espacio columna de A (direcciones de salida).
*   Las **filas de Vᵀ** (o columnas de V) son una base ortonormal para el espacio fila de A (direcciones de entrada).
*   Los **valores singulares** indican la importancia ("fuerza") de cada dirección: el primer valor singular está asociado a la dirección más importante.

La SVD descompone la transformación lineal en tres pasos:
1. Rotación en el espacio de entrada (Vᵀ).
2. Escalado por los valores singulares (Σ).
3. Rotación en el espacio de salida (U).

In [ ]:
# Visualización de los valores singulares
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.bar(range(1, len(s)+1), s, color='blue', alpha=0.7)
plt.xlabel('Índice')
plt.ylabel('Valor singular')
plt.title('Valores singulares de A')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(np.cumsum(s**2) / np.sum(s**2), 'ro-', linewidth=2)
plt.xlabel('Número de componentes')
plt.ylabel('Proporción de varianza acumulada')
plt.title('Varianza explicada por los valores singulares')
plt.grid(True)

plt.tight_layout()
plt.show()

---
## 2. Aproximación de Bajo Rango

Podemos aproximar $\mathbf{A}$ usando solo los $k$ primeros valores singulares:

$$\mathbf{A}_k = \sum_{i=1}^k \sigma_i \mathbf{u}_i \mathbf{v}_i^\top = \mathbf{U}_k \boldsymbol{\Sigma}_k \mathbf{V}_k^\top$$

Esta es la mejor aproximación de rango $k$ a $\mathbf{A}$ en normas espectral y de Frobenius (Teorema de Eckart-Young).

In [ ]:
# Función para aproximar A con rango k
def low_rank_approx(A, k):
    U, s, Vt = np.linalg.svd(A, full_matrices=False)
    return U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]

# Probamos con diferentes k
k_values = [1, 2, 3]

print("🔷 Aproximaciones de bajo rango de A:")
for k in k_values:
    A_k = low_rank_approx(A, k)
    error = np.linalg.norm(A - A_k, 'fro')
    print(f"  k={k}, error Frobenius: {error:.4f}")
    print(A_k.round(2))
    print()

---
## 3. Aplicación: Compresión de Imágenes con SVD

Una imagen en escala de grises es una matriz de píxeles. Podemos comprimirla truncando su SVD.

In [ ]:
# Cargamos una imagen de ejemplo (cámara)
image = data.camera().astype(float) / 255.0
h, w = image.shape
print(f"Imagen original: {h} x {w} píxeles")

# Aplicamos SVD a la imagen
U_img, s_img, Vt_img = np.linalg.svd(image, full_matrices=False)

# Visualizamos los valores singulares
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(s_img, 'b-', linewidth=1)
plt.yscale('log')
plt.xlabel('Índice')
plt.ylabel('Valor singular (escala log)')
plt.title('Valores singulares de la imagen')
plt.grid(True)

plt.subplot(1, 2, 2)
cum_var = np.cumsum(s_img**2) / np.sum(s_img**2)
plt.plot(cum_var, 'r-', linewidth=2)
plt.xlabel('Número de componentes')
plt.ylabel('Varianza acumulada')
plt.title('Varianza explicada')
plt.grid(True)
plt.axhline(y=0.9, color='g', linestyle='--', label='90%')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Comprimimos la imagen con diferentes números de componentes
k_values = [5, 20, 50, 100]

plt.figure(figsize=(15, 8))

# Mostramos la original
plt.subplot(2, len(k_values)+1, 1)
plt.imshow(image, cmap='gray')
plt.title('Original')
plt.axis('off')

# Para cada k, mostramos la imagen comprimida y el error
for i, k in enumerate(k_values):
    # Reconstrucción con k componentes
    img_k = U_img[:, :k] @ np.diag(s_img[:k]) @ Vt_img[:k, :]
    img_k = np.clip(img_k, 0, 1)  # asegurar rango
    
    plt.subplot(2, len(k_values)+1, i+2)
    plt.imshow(img_k, cmap='gray')
    plt.title(f'k={k}')
    plt.axis('off')
    
    plt.subplot(2, len(k_values)+1, i+len(k_values)+2)
    error = np.linalg.norm(image - img_k, 'fro') / np.linalg.norm(image, 'fro')
    plt.bar(i, error, color='red', alpha=0.7)
    plt.xticks([])
    plt.ylim(0, 0.3)
    plt.ylabel('Error relativo')

plt.suptitle('Compresión de imagen mediante SVD', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

# Cálculo de ratios de compresión
original_size = h * w
for k in k_values:
    compressed_size = k * (h + w + 1)  # U[:,:k] (h*k) + s[:k] (k) + Vt[:k,:] (k*w)
    ratio = compressed_size / original_size
    print(f"k={k}: tamaño comprimido = {compressed_size}, ratio = {ratio:.4f}")

---
## 4. Conexiones IA

### 4.1. ML/CV: Compresión de imágenes y reducción de ruido
Como hemos visto, la SVD permite comprimir imágenes eliminando componentes de baja energía, que suelen corresponder a ruido.

### 4.2. RecSys: Filtrado colaborativo
La matriz usuario-producto (con calificaciones) se aproxima mediante SVD de bajo rango para predecir ratings faltantes.

### 4.3. NLP: Latent Semantic Analysis (LSA)
La matriz término-documento (tf-idf) se descompone vía SVD para encontrar "conceptos" latentes.

### 4.4. DL: Pseudoinversa e inicialización
La pseudoinversa $\mathbf{A}^+ = \mathbf{V} \boldsymbol{\Sigma}^+ \mathbf{U}^\top$ se usa en mínimos cuadrados y en inicialización de pesos.

### 4.5. GenAI: Análisis del espacio latente
En modelos generativos, la SVD del espacio latente puede revelar direcciones interpretables (por ejemplo, "sonrisa" en un modelo de caras).

In [ ]:
# Ejemplo: Pseudoinversa con SVD
# Resolver un sistema Ax = b con mínimos cuadrados
m, n = 10, 5
A_pinv = np.random.randn(m, n)
x_true = np.random.randn(n)
b = A_pinv @ x_true + 0.1 * np.random.randn(m)

# Pseudoinversa vía SVD
U, s, Vt = np.linalg.svd(A_pinv, full_matrices=False)
s_inv = np.diag(1.0 / s)
x_pinv = Vt.T @ s_inv @ U.T @ b

# Comparación con lstsq
x_lstsq, _, _, _ = np.linalg.lstsq(A_pinv, b, rcond=None)

print("🔷 Pseudoinversa:")
print(f"  Error con verdadero (pinv): {np.linalg.norm(x_pinv - x_true):.4f}")
print(f"  Error con verdadero (lstsq): {np.linalg.norm(x_lstsq - x_true):.4f}")

In [ ]:
# Simulación conceptual de filtrado colaborativo
# Creamos una matriz de calificaciones con estructura de bajo rango
n_users, n_items, k_true = 50, 40, 3
U_true = np.random.randn(n_users, k_true)
V_true = np.random.randn(n_items, k_true)
R = U_true @ V_true.T + 0.2 * np.random.randn(n_users, n_items)

# Aplicamos SVD y truncamos a k componentes
U_r, s_r, Vt_r = np.linalg.svd(R, full_matrices=False)
k = 3
R_pred = U_r[:, :k] @ np.diag(s_r[:k]) @ Vt_r[:k, :]

# Error de aproximación
print(f"Error de aproximación con rango {k}: {np.linalg.norm(R - R_pred, 'fro'):.4f}")

---
## Ejercicios para el Estudiante

1.  **SVD de una imagen propia:** Sube tu propia imagen en escala de grises y aplica compresión SVD con diferentes k. ¿Cuál es el mínimo k para que la imagen sea reconocible?

2.  **Ruido y SVD:** Añade ruido gaussiano a la imagen de cámara y luego aplica SVD truncado. ¿El truncamiento ayuda a eliminar el ruido?

3.  **Pseudoinversa vs solve:** Para un sistema cuadrado bien condicionado, compara la solución usando pseudoinversa, `np.linalg.solve` e inversa directa. ¿Cuál es más rápido?

4.  **SVD en color:** Aplica SVD a cada canal (R, G, B) de una imagen a color por separado y reconstruye combinando los canales. Compara la calidad visual.

5.  **Reflexión:** ¿Por qué la SVD se considera más estable numéricamente que la descomposición por autovalores para matrices no cuadradas o mal condicionadas?

---
## Conclusiones de la Sesión

*   La **SVD** factoriza cualquier matriz $\mathbf{A} = \mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^\top$, donde $\mathbf{U}$ y $\mathbf{V}$ son ortogonales y $\boldsymbol{\Sigma}$ contiene los valores singulares.
*   Los **valores singulares** indican la importancia de cada dirección; los primeros capturan la mayor parte de la información.
*   La **aproximación de bajo rango** $\mathbf{A}_k$ (truncando la SVD) es la mejor aproximación de rango $k$ a $\mathbf{A}$.
*   Aplicamos SVD para **comprimir imágenes**, reduciendo drásticamente el almacenamiento con pérdida controlada de calidad.
*   Conectamos la SVD con aplicaciones clave en IA:
    *   **Visión:** compresión y reducción de ruido.
    *   **Sistemas de recomendación:** filtrado colaborativo.
    *   **NLP:** Latent Semantic Analysis (LSA).
    *   **Deep Learning:** pseudoinversa e inicialización de pesos.
    *   **Generative AI:** análisis del espacio latente.

¡La SVD es verdaderamente la navaja suiza del álgebra lineal!